# Paredes por estilo

> **Qué compara.** En FloorPlanCAD muchas paredes tienen líneas adentro: el eje, capas o un rayado (70 de las 185 de la muestra de abajo). En BLD-AR casi todas son una doble línea vacía. Si el modelo aprendió a reconocer las paredes por esas líneas, va a encontrar peor las vacías, y eso explicaría parte de su caída en BLD-AR.
>
> **Cómo lo mide.**
> 1. Toma una muestra de paredes del test de FloorPlanCAD, marcadas a ojo según tengan o no una línea adentro.
> 2. Pasa el modelo por los planos de esas paredes, igual que en la evaluación en BLD-AR, y mide qué parte de cada pared marca. También calcula el AP50 de seg de pared de cada grupo.
> 3. Compara las paredes sin líneas con las que tienen líneas. Se hace dentro del test de FloorPlanCAD, porque ahí solo cambia el estilo y no el conjunto.
>
> **Cómo se decide.** Se calculan dos promedios: A, qué parte de cada pared sin líneas marca el modelo, y B, qué parte de cada pared con líneas. Después se divide A / B. Por ejemplo, si a las paredes sin líneas les marca el 60 % y a las con líneas el 90 %, A / B = 60 / 90 = 67 %.
>
> | A / B | Qué quiere decir | Hipótesis |
> |---|---|---|
> | Menos del 80 % | Encuentra claramente peor las paredes sin líneas | Confirmada |
> | Entre 80 % y 95 % | Las encuentra algo peor | Confirmada en parte |
> | 95 % o más | Las encuentra casi igual: las líneas no importan | Descartada |

In [ ]:
!nvidia-smi
!for pid in $(nvidia-smi --query-compute-apps=pid --format=csv,noheader); do ps -o user=,pid=,etime=,cmd= -p $pid; done

In [ ]:
%pip install ultralytics==8.4.173 sahi==0.12.8 pycocotools==2.0.11

## Imports

In [ ]:
from pathlib import Path

import numpy as np
import shapely
import yaml
from IPython.display import Markdown, display
from PIL import Image, ImageDraw
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from shapely.geometry import Polygon

## Rutas y parámetros

In [ ]:
REPO = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "datasets").is_dir()), Path.cwd())
FLOORPLANCAD = REPO / "datasets/floor-plan-cad-yolo"
WEIGHTS = REPO / "weights/00-YOLO26m-SEG-1024/best.pt"

TILE = 1024
OVERLAP = 0.2
CONF = 0.05        # como en la evaluación en BLD-AR, para el AP50 de seg
PIXEL_CONF = 0.25  # desde esta confianza una predicción marca pared, para la pared encontrada
DEVICE = "cuda:0"

assert WEIGHTS.exists(), f"No existen los pesos: {WEIGHTS}"

## Muestra marcada a ojo

> 300 paredes elegidas al azar entre las 14 040 del test de FloorPlanCAD (semilla 0) y marcadas a ojo:
> - **tiene:** hay una línea a lo largo de la pared (eje, capa o rayado), aunque sea punteada;
> - **no tiene:** no hay líneas a lo largo. No cuentan los bordes ni las líneas que la cruzan de lado a lado;
> - **no se ve:** la etiqueta no coincide con el dibujo, la pared es un cruce de líneas o el caso es dudoso.
>
> Acá están solo las 185 que tienen o no tienen líneas (70 y 115); las 115 "no se ve" quedan afuera. Cada fila es el plano, el número de línea de la pared en su archivo de etiquetas y la marca.

In [ ]:
MARKS = [
    ("test-00_0056-0061", 5, "no tiene"),
    ("test-00_0058-0013", 16, "tiene"),
    ("test-00_0073-0022", 3, "tiene"),
    ("test-00_0108-0026", 15, "no tiene"),
    ("test-00_0132-0011", 16, "tiene"),
    ("test-00_0153-0007", 12, "tiene"),
    ("test-00_0178-0009", 9, "tiene"),
    ("test-00_0193-0007", 68, "no tiene"),
    ("test-00_0193-0007", 80, "no tiene"),
    ("test-00_0193-0007", 98, "no tiene"),
    ("test-00_0200-0002", 14, "tiene"),
    ("test-00_0218-0019", 13, "tiene"),
    ("test-00_0311-0007", 13, "no tiene"),
    ("test-00_0316-0017", 10, "tiene"),
    ("test-00_0316-0017", 23, "tiene"),
    ("test-00_0352-0028", 22, "no tiene"),
    ("test-00_0398-0018", 19, "tiene"),
    ("test-00_0429-0011", 3, "no tiene"),
    ("test-00_0432-0025", 9, "no tiene"),
    ("test-00_0521-0010", 13, "no tiene"),
    ("test-00_0569-0026", 17, "no tiene"),
    ("test-00_0569-0026", 18, "no tiene"),
    ("test-00_0569-0026", 22, "no tiene"),
    ("test-00_0569-0026", 31, "no tiene"),
    ("test-00_0591-0931", 5, "tiene"),
    ("test-00_0640-0007", 32, "tiene"),
    ("test-00_0652-0070", 13, "tiene"),
    ("test-00_0683-0225", 11, "no tiene"),
    ("test-00_0767-4353", 3, "no tiene"),
    ("test-00_0792-0012", 12, "no tiene"),
    ("test-00_0797-2143", 3, "no tiene"),
    ("test-00_0800-4461", 16, "no tiene"),
    ("test-00_0823-0018", 9, "tiene"),
    ("test-00_0851-0619", 3, "no tiene"),
    ("test-00_0894-0008", 10, "tiene"),
    ("test-00_0966-0987", 5, "no tiene"),
    ("test-00_0970-0010", 4, "no tiene"),
    ("test-00_0975-0050", 18, "no tiene"),
    ("test-00_0980-0008", 10, "no tiene"),
    ("test-00_0999-0004", 25, "tiene"),
    ("test-00_1009-0006", 18, "no tiene"),
    ("test-00_1029-0002", 7, "no tiene"),
    ("test-00_1031-0012", 32, "no tiene"),
    ("test-00_1064-0016", 1, "no tiene"),
    ("test-00_1094-0146", 3, "no tiene"),
    ("test-00_1107-0029", 17, "tiene"),
    ("test-00_1107-0029", 23, "no tiene"),
    ("test-00_1117-0089", 39, "no tiene"),
    ("test-00_1117-0100", 9, "no tiene"),
    ("test-00_1127-0004", 8, "tiene"),
    ("test-00_1213-0057", 17, "no tiene"),
    ("test-00_1253-0008", 8, "no tiene"),
    ("test-00_1291-0022", 27, "no tiene"),
    ("test-00_1291-0022", 34, "no tiene"),
    ("test-00_1316-0021", 11, "no tiene"),
    ("test-00_1316-0049", 14, "no tiene"),
    ("train-00_0027-0079", 12, "no tiene"),
    ("train-00_0094-0008", 3, "tiene"),
    ("train-00_0107-0048", 31, "no tiene"),
    ("train-00_0119-0019", 13, "no tiene"),
    ("train-00_0129-0007", 11, "no tiene"),
    ("train-00_0136-0017", 1, "tiene"),
    ("train-00_0144-0001", 11, "no tiene"),
    ("train-00_0144-0001", 14, "no tiene"),
    ("train-00_0144-0001", 21, "no tiene"),
    ("train-00_0156-0012", 17, "no tiene"),
    ("train-00_0180-0013", 19, "tiene"),
    ("train-00_0185-0016", 10, "tiene"),
    ("train-00_0190-0007", 21, "no tiene"),
    ("train-00_0190-0013", 11, "tiene"),
    ("train-00_0191-0016", 8, "no tiene"),
    ("train-00_0226-0001", 49, "tiene"),
    ("train-00_0230-0002", 17, "no tiene"),
    ("train-00_0240-0035", 41, "no tiene"),
    ("train-00_0247-0003", 15, "tiene"),
    ("train-00_0265-0003", 28, "tiene"),
    ("train-00_0285-0002", 4, "no tiene"),
    ("train-00_0285-0002", 5, "no tiene"),
    ("train-00_0304-0040", 8, "no tiene"),
    ("train-00_0324-0010", 13, "no tiene"),
    ("train-00_0332-0677", 8, "no tiene"),
    ("train-00_0358-0009", 12, "no tiene"),
    ("train-00_0378-0013", 7, "no tiene"),
    ("train-00_0387-0052", 34, "no tiene"),
    ("train-00_0389-0048", 10, "no tiene"),
    ("train-00_0394-0075", 7, "no tiene"),
    ("train-00_0396-0028", 7, "no tiene"),
    ("train-00_0436-0009", 16, "tiene"),
    ("train-00_0445-0043", 30, "tiene"),
    ("train-00_0446-0039", 21, "no tiene"),
    ("train-00_0464-0007", 28, "tiene"),
    ("train-00_0464-0007", 36, "tiene"),
    ("train-00_0468-0002", 21, "tiene"),
    ("train-00_0479-0003", 15, "no tiene"),
    ("train-00_0506-0008", 27, "no tiene"),
    ("train-00_0518-0010", 24, "tiene"),
    ("train-00_0569-0007", 8, "no tiene"),
    ("train-01_0020-0007", 11, "tiene"),
    ("train-01_0024-0037", 13, "tiene"),
    ("train-01_0077-0049", 7, "no tiene"),
    ("train-01_0100-0011", 13, "no tiene"),
    ("train-01_0118-0023", 17, "tiene"),
    ("train-01_0133-0011", 15, "no tiene"),
    ("train-01_0134-0051", 5, "no tiene"),
    ("train-01_0141-0011", 18, "tiene"),
    ("train-01_0141-0014", 23, "tiene"),
    ("train-01_0141-0014", 25, "tiene"),
    ("train-01_0163-0006", 15, "tiene"),
    ("train-01_0164-0017", 28, "tiene"),
    ("train-01_0168-0025", 2, "no tiene"),
    ("train-01_0196-0005", 45, "no tiene"),
    ("train-01_0257-0633", 0, "no tiene"),
    ("train-01_0278-0004", 6, "no tiene"),
    ("train-01_0300-0084", 10, "tiene"),
    ("train-01_0326-0173", 23, "no tiene"),
    ("train-01_0349-0599", 8, "no tiene"),
    ("train-01_0363-0002", 7, "no tiene"),
    ("train-01_0440-0024", 17, "no tiene"),
    ("train-01_0440-0024", 20, "no tiene"),
    ("train-01_0440-0024", 25, "tiene"),
    ("train-01_0487-0017", 14, "tiene"),
    ("train-01_0487-0017", 18, "no tiene"),
    ("train-01_0492-0001", 1, "no tiene"),
    ("train-01_0493-0238", 7, "no tiene"),
    ("train-01_0493-0292", 7, "no tiene"),
    ("train-01_0497-0006", 22, "no tiene"),
    ("train-01_0498-0108", 17, "tiene"),
    ("train-01_0508-0011", 3, "tiene"),
    ("train-01_0522-0015", 5, "tiene"),
    ("train-01_0574-0006", 5, "tiene"),
    ("train-01_0591-0844", 9, "no tiene"),
    ("train-01_0591-0935", 9, "tiene"),
    ("train-01_0618-0027", 28, "tiene"),
    ("train-01_0618-0027", 41, "tiene"),
    ("train-01_0662-0006", 20, "no tiene"),
    ("train-01_0662-0021", 40, "no tiene"),
    ("train-01_0671-0013", 6, "no tiene"),
    ("train-01_0671-0019", 34, "tiene"),
    ("train-01_0674-0730", 5, "no tiene"),
    ("train-01_0682-1162", 7, "no tiene"),
    ("train-01_0699-0046", 21, "no tiene"),
    ("train-01_0719-0051", 23, "no tiene"),
    ("train-01_0724-0028", 5, "tiene"),
    ("train-01_0726-0459", 1, "no tiene"),
    ("train-01_0736-0042", 24, "no tiene"),
    ("train-01_0764-0007", 15, "tiene"),
    ("train-01_0789-0022", 26, "tiene"),
    ("train-01_0789-0022", 27, "tiene"),
    ("train-01_0797-2256", 6, "no tiene"),
    ("train-01_0797-2554", 5, "no tiene"),
    ("train-01_0815-0067", 12, "tiene"),
    ("train-01_0817-0004", 25, "tiene"),
    ("train-01_0817-0004", 27, "tiene"),
    ("train-01_0857-0015", 6, "no tiene"),
    ("train-01_0908-0136", 2, "no tiene"),
    ("train-01_0912-0005", 1, "no tiene"),
    ("train-01_0913-0001", 10, "tiene"),
    ("train-01_0942-0010", 19, "no tiene"),
    ("train-01_1003-0011", 0, "tiene"),
    ("train-01_1027-0081", 12, "no tiene"),
    ("train-01_1121-0057", 37, "no tiene"),
    ("train-01_1141-0002", 16, "tiene"),
    ("train-01_1148-0029", 49, "no tiene"),
    ("train-01_1163-3821", 10, "tiene"),
    ("train-01_1181-0030", 8, "tiene"),
    ("train-01_1188-0288", 8, "no tiene"),
    ("train-01_1213-0059", 41, "no tiene"),
    ("train-01_1213-0059", 64, "no tiene"),
    ("train-01_1224-0017", 36, "no tiene"),
    ("train-01_1241-0018", 13, "tiene"),
    ("train-01_1247-0009", 15, "tiene"),
    ("train-01_1247-0009", 22, "tiene"),
    ("train-01_1252-0026", 12, "no tiene"),
    ("train-01_1252-0031", 18, "no tiene"),
    ("train-01_1262-0612", 15, "no tiene"),
    ("train-01_1288-0007", 19, "tiene"),
    ("train-01_1288-0007", 48, "tiene"),
    ("train-01_1289-0004", 14, "no tiene"),
    ("train-01_1289-0004", 22, "no tiene"),
    ("train-01_1296-0006", 14, "no tiene"),
    ("train-01_1300-0001", 2, "no tiene"),
    ("train-01_1300-0001", 8, "tiene"),
    ("train-01_1307-0026", 13, "no tiene"),
    ("train-01_1312-0085", 5, "no tiene"),
    ("train-01_1327-0066", 17, "tiene"),
]

## Modelo

In [ ]:
model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics", model_path=str(WEIGHTS),
    confidence_threshold=CONF, device=DEVICE, image_size=TILE,
)

## Formas y máscaras

In [ ]:
def to_geometry(points):
    """Polígono válido a partir de sus puntos; puede quedar vacío o partido en varios."""
    if len(points) < 3:
        return Polygon()
    parts = [g for g in shapely.get_parts(shapely.make_valid(Polygon(points))) if g.geom_type == "Polygon"]
    return shapely.unary_union(parts) if parts else Polygon()


def polygons(geometry):
    return [g for g in shapely.get_parts(geometry) if g.geom_type == "Polygon" and not g.is_empty]


def region_mask(geometry, box):
    """Máscara de una geometría dentro del recuadro box = (x0, y0, x1, y1) del plano."""
    canvas = Image.new("1", (box[2] - box[0], box[3] - box[1]))
    draw = ImageDraw.Draw(canvas)
    for part in polygons(geometry):
        draw.polygon([(x - box[0], y - box[1]) for x, y in part.exterior.coords], fill=1)
    return np.array(canvas)


def clipped_box(geometry, size):
    x0, y0, x1, y1 = geometry.bounds
    return max(int(x0), 0), max(int(y0), 0), min(int(x1) + 1, size[0]), min(int(y1) + 1, size[1])

## Pared encontrada en cada plano

> Las paredes del plano que no están en la muestra se guardan como zonas a ignorar para el AP50 de seg.

In [ ]:
def predicted_walls(image_path):
    """Predicciones de wall por tiles, con los mismos parámetros que la evaluación en BLD-AR."""
    result = get_sliced_prediction(
        str(image_path), model,
        slice_height=TILE, slice_width=TILE,
        overlap_height_ratio=OVERLAP, overlap_width_ratio=OVERLAP,
        perform_standard_pred=False, verbose=0,
        # Con confianza < 0.1 SAHI cambia a NMS y no une los muros cortados entre tiles
        force_postprocess_type=True,
    )
    return [p for p in result.to_coco_predictions() if p["category_name"] == "wall"]


names = yaml.safe_load((FLOORPLANCAD / "floor-plan-cad-yolo.yaml").read_text())["names"]
wall_id = next(i for i, name in names.items() if name == "wall")
images, sampled, ignored, predictions = [], [], [], []

for plan in sorted({plan for plan, _, _ in MARKS}):
    image_path = FLOORPLANCAD / "images/test" / f"{plan}.png"
    size = Image.open(image_path).size
    image_id = len(images) + 1
    images.append({"id": image_id, "width": size[0], "height": size[1]})
    plan_marks = {line: mark for p, line, mark in MARKS if p == plan}

    plan_predictions = predicted_walls(image_path)
    marked = Image.new("1", size)
    draw = ImageDraw.Draw(marked)
    for prediction in plan_predictions:
        if prediction["score"] >= PIXEL_CONF:
            for polygon in prediction["segmentation"]:
                draw.polygon(polygon, fill=1)
    marked = np.array(marked)
    predictions += [{**p, "image_id": image_id, "category_id": 1} for p in plan_predictions]

    # Etiqueta YOLO: clase y puntos x, y entre 0 y 1
    rows = [line.split() for line in (FLOORPLANCAD / "labels/test" / f"{plan}.txt").read_text().splitlines()]
    for line, row in enumerate(rows):
        if int(row[0]) != wall_id:
            continue
        wall = to_geometry(np.reshape(np.array(row[1:], dtype=float), (-1, 2)) * size)
        if wall.is_empty:
            continue
        if line in plan_marks:
            box = clipped_box(wall, size)
            mask = region_mask(wall, box)
            found = (marked[box[1]:box[3], box[0]:box[2]] & mask).sum() / mask.sum() if mask.any() else np.nan
            sampled.append((image_id, wall, plan_marks[line], found))
        else:
            ignored.append((image_id, wall))

## AP50 de seg de pared de cada grupo

> Las paredes del otro grupo y las que no están en la muestra son zonas a ignorar: lo que el modelo predice sobre ellas no cuenta como acierto ni como error.

In [ ]:
def coco_annotation(geometry, image_id, iscrowd):
    x0, y0, x1, y1 = geometry.bounds
    return {
        "image_id": image_id, "category_id": 1, "iscrowd": iscrowd,
        "segmentation": [np.ravel(part.exterior.coords).tolist() for part in polygons(geometry)],
        "bbox": [x0, y0, x1 - x0, y1 - y0], "area": geometry.area,
    }


def ap50(mark):
    """AP50 de seg de pared de un grupo, con COCOeval como en la evaluación en BLD-AR."""
    annotations = [coco_annotation(wall, image_id, iscrowd=int(wall_mark != mark))
                   for image_id, wall, wall_mark, _ in sampled]
    annotations += [coco_annotation(wall, image_id, iscrowd=1) for image_id, wall in ignored]
    for annotation_id, annotation in enumerate(annotations, start=1):
        annotation["id"] = annotation_id
    coco_gt = COCO()
    coco_gt.dataset = {"images": images, "annotations": annotations, "categories": [{"id": 1, "name": "wall"}]}
    coco_gt.createIndex()
    # Copias: COCOeval reemplaza los polígonos de las predicciones por RLE
    evaluator = COCOeval(coco_gt, coco_gt.loadRes([dict(p) for p in predictions]), "segm")
    evaluator.evaluate()
    evaluator.accumulate()
    precision = evaluator.eval["precision"][0, :, 0, 0, -1]  # IoU 0.5, área "all", 100 detecciones
    return precision[precision > -1].mean()

## Resultado

> Pared encontrada: qué parte de cada pared marca el modelo, en promedio por pared.

In [ ]:
found = {mark: [value for _, _, wall_mark, value in sampled if wall_mark == mark] for mark in ("no tiene", "tiene")}
a, b = np.nanmean(found["no tiene"]), np.nanmean(found["tiene"])

table = "| Paredes | Cantidad | Pared encontrada | AP50 de seg de pared |\n|---|---|---|---|\n"
table += f"| Sin líneas adentro (A) | {len(found['no tiene'])} | {a:.3f} | {ap50('no tiene'):.3f} |\n"
table += f"| Con líneas adentro (B) | {len(found['tiene'])} | {b:.3f} | {ap50('tiene'):.3f} |\n"
display(Markdown(
    f"**{WEIGHTS.parent.name} · test de FloorPlanCAD**\n\n" + table
    + f"\nEl modelo marca en promedio el {a:.0%} de cada pared sin líneas (A) y el {b:.0%} de cada pared con líneas (B): "
    f"A / B = **{a / b:.0%}**."))